# Ação + put de proteção, com dados reais do Yahoo Finance

## O que é a estratégia
Você **compra a ação** e, junto, **compra uma put** (opção de venda) sobre ela. A put dá o direito de **vender a ação por um preço fixo, a strike**, até o vencimento. Se o mercado cair, você vende pela strike, não pelo preço de mercado. Se o mercado subir, a put perde o valor e você fica com a alta da ação, descontado o que pagou pela proteção.

Por ação, as contas são:

| Conta | Fórmula | O que significa |
|---|---|---|
| Custo total | preço da ação + prêmio da put + taxas | quanto você desembolsa |
| **Preço de venda garantido** | strike da put | o valor mínimo pelo qual você consegue vender a ação até o vencimento |
| **Perda máxima** | custo total − strike | o pior resultado possível, mesmo se a ação for a zero |
| **Break-even** | custo total ÷ (1 − taxa de venda) | o preço que a ação precisa ter no vencimento para você sair sem perda; acima dele há lucro |
| Custo da proteção | prêmio ÷ preço da ação | quanto a proteção "come" do seu retorno |

## O que o notebook faz
1. **Baixa do Yahoo Finance** o histórico de preços, os dividendos, a taxa de juros americana (`^IRX`) e a **cadeia real de puts** de cada ativo.
2. Para cada strike líquida, monta o **menu de proteção** com as contas acima.
3. Mede o **risco com Monte Carlo** (modelo GARCH-t ajustado ao histórico real): probabilidade de perda, VaR e Expected Shortfall, com e sem a put.
4. **Recomenda uma strike** por ativo: a de menor custo líquido esperado entre as que respeitam a sua perda máxima aceitável.
5. Usa uma **Deep Q-Network (DQN)** treinada no histórico real do ativo para indicar se hoje é um bom dia de entrada ou se vale esperar alguns dias. A DQN é avaliada em um período que ela nunca viu, contra regras simples.
6. Junta tudo num **painel final**.

## Fonte dos dados e limites
- **Tudo vem do Yahoo Finance**, sem preços inventados. Se o Yahoo não tiver opções para um ativo, ele aparece como **não elegível**, com o motivo.
- O Yahoo publica cadeias de opções na prática **só para ativos listados nos EUA**. Ações da B3 (final `.SA`) em geral ficam sem opções.
- As cotações têm cerca de **15 minutos de atraso**. **Fora do pregão** o Yahoo costuma zerar bid e ask; nesse caso o notebook usa o último negócio e avisa. Rode durante o pregão para ter preços executáveis.

> Nada aqui garante lucro e não é recomendação de investimento. Confira os preços na corretora antes de operar.

## 1. Configuração

Todos os parâmetros ficam em um único objeto, `AnalysisSettings`. Para mudar algo, edite o valor padrão ou crie o objeto com outros valores, por exemplo `AnalysisSettings(tickers=("AAPL", "KO"))`.

In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class AnalysisSettings:
    # Ativos analisados (formato do Yahoo Finance). Opções do Yahoo: na prática, só ativos dos EUA.
    tickers: tuple = ("AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "JPM", "XOM", "KO", "SPY")
    history_start_date: str = "2012-01-01"

    # Tamanho da posição: cada contrato de put protege 100 ações
    contracts_per_ticker: int = 1                # posição padrão = 1 contrato = 100 ações
    shares_per_ticker: dict = field(default_factory=dict)   # quantidade específica, ex.: {"AAPL": 300} (múltiplo de 100)
    max_capital_per_ticker: float = None         # se definido, ativos que exigem mais capital ficam de fora
    shares_per_option_contract: int = 100

    # Proteção
    target_protection_days: int = 45            # prazo desejado da put (dias corridos)
    minimum_protection_days: int = 14           # descarta vencimentos mais curtos que isso
    strike_range_vs_price: tuple = (0.80, 1.02)  # strikes avaliadas, em % do preço atual
    max_acceptable_loss_pct: float = 0.08       # perda máxima aceitável na posição protegida

    # Liquidez mínima das puts
    minimum_open_interest: int = 100
    maximum_bid_ask_spread_pct: float = 0.20    # (ask − bid) / preço médio

    # Custos
    stock_commission_pct: float = 0.0005        # por lado (compra e venda da ação)
    option_commission_per_contract: float = 0.65

    # Monte Carlo
    monte_carlo_paths: int = 20_000

    # DQN para ponto de entrada
    entry_window_days: int = 10                 # dias úteis que você aceita esperar para comprar
    dqn_training_rounds: int = 120              # rodadas de treino (64 episódios em paralelo por rodada)
    dqn_hidden_units: int = 64
    test_period_fraction: float = 0.30          # fração final do histórico reservada para teste

    random_seed: int = 42


SETTINGS = AnalysisSettings()
SETTINGS

## 2. Bibliotecas e estilo dos gráficos

In [ ]:
import copy
import logging
import warnings
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter
from scipy import optimize, signal, stats
from scipy.special import gammaln

warnings.filterwarnings("ignore")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

COLORS = {"stock": "#2a78d6", "protected": "#eb6834", "accent": "#1baf7a", "warning": "#eda100",
          "ink": "#52514e", "ink_strong": "#0b0b0b", "grid": "#e6e5e1", "surface": "#fcfcfb",
          "loss": "#d03b3b", "muted": "#a3a29c"}
plt.rcParams.update({
    "figure.facecolor": COLORS["surface"], "axes.facecolor": COLORS["surface"], "savefig.facecolor": COLORS["surface"],
    "axes.edgecolor": COLORS["grid"], "axes.labelcolor": COLORS["ink"], "xtick.color": COLORS["ink"],
    "ytick.color": COLORS["ink"], "text.color": COLORS["ink_strong"], "axes.grid": True, "grid.color": COLORS["grid"],
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2, "figure.figsize": (10, 4.4),
    "font.size": 10, "axes.titleweight": "bold", "axes.titlesize": 11, "legend.frameon": False,
})
TODAY = pd.Timestamp.today().normalize()
TRADING_DAYS_PER_YEAR = 252

## 3. Acesso aos dados do Yahoo Finance

Cada função busca **uma** informação e devolve um valor simples. Se o Yahoo não tiver o dado, a função lança `MarketDataError` com uma mensagem clara. Essa mensagem vira o **motivo de não elegibilidade** do ativo.

In [ ]:
class MarketDataError(Exception):
    # Dado indisponível no Yahoo Finance.
    pass


def download_price_history(ticker: str, start_date: str) -> pd.Series:
    # Fechamentos diários ajustados por dividendos e desdobramentos.
    history = yf.Ticker(ticker).history(start=start_date, auto_adjust=True)
    if history is None or history.empty:
        raise MarketDataError("sem histórico de preços no Yahoo")
    closes = history["Close"].dropna()
    closes.index = pd.to_datetime(closes.index).tz_localize(None)
    return closes[closes > 0]


def get_current_stock_price(ticker: str, fallback_price: float) -> float:
    # Último preço negociado (fast_info); se indisponível, o último fechamento.
    try:
        last_price = float(yf.Ticker(ticker).fast_info["lastPrice"])
        return last_price if last_price > 0 else fallback_price
    except Exception:
        return fallback_price


def get_trailing_dividend_yield(ticker: str, current_price: float) -> float:
    # Dividendos pagos nos últimos 12 meses ÷ preço atual.
    dividends = yf.Ticker(ticker).dividends
    if dividends is None or dividends.empty:
        return 0.0
    dividends.index = pd.to_datetime(dividends.index).tz_localize(None)
    last_year_total = dividends[dividends.index >= TODAY - pd.Timedelta(days=365)].sum()
    return float(last_year_total / current_price)


def get_us_risk_free_rate() -> float:
    # Taxa do Treasury de 13 semanas (^IRX), em fração ao ano.
    history = yf.Ticker("^IRX").history(period="1mo")
    if history is None or history.empty:
        raise MarketDataError("sem taxa ^IRX no Yahoo")
    return float(history["Close"].dropna().iloc[-1]) / 100


def list_option_expirations(ticker: str) -> list:
    # Vencimentos de opções disponíveis no Yahoo.
    expirations = yf.Ticker(ticker).options
    if not expirations:
        raise MarketDataError("o Yahoo não tem opções para este ativo")
    return [pd.Timestamp(date) for date in expirations]


def download_put_chain(ticker: str, expiration: pd.Timestamp) -> pd.DataFrame:
    # Todas as puts de um vencimento, como o Yahoo publica.
    chain = yf.Ticker(ticker).option_chain(expiration.strftime("%Y-%m-%d"))
    puts = chain.puts.copy()
    if puts.empty:
        raise MarketDataError(f"sem puts para o vencimento {expiration.date()}")
    return puts

## 4. Escolha do vencimento e limpeza da cadeia de puts

1. **Vencimento:** o mais próximo de `target_protection_days`, com pelo menos `minimum_protection_days`.
2. **Strikes:** só as que ficam entre 80% e 102% do preço atual (ajustável em `strike_range_vs_price`).
3. **Preço da put:** o **ask**, que é o que você paga para comprar. Fora do pregão, quando bid e ask vêm zerados, usa o último negócio e marca a origem como `último negócio`.
4. **Liquidez:** exige um mínimo de contratos em aberto e, quando há cotação ao vivo, um spread máximo.

In [ ]:
def choose_expiration(expirations: list, settings: AnalysisSettings) -> pd.Timestamp:
    long_enough = [date for date in expirations if (date - TODAY).days >= settings.minimum_protection_days]
    if not long_enough:
        raise MarketDataError("nenhum vencimento com prazo mínimo")
    return min(long_enough, key=lambda date: abs((date - TODAY).days - settings.target_protection_days))


def prepare_liquid_puts(raw_puts: pd.DataFrame, stock_price: float, settings: AnalysisSettings) -> pd.DataFrame:
    puts = raw_puts.copy()
    lowest_strike, highest_strike = (stock_price * bound for bound in settings.strike_range_vs_price)
    puts = puts[(puts["strike"] >= lowest_strike) & (puts["strike"] <= highest_strike)]

    has_live_quote = (puts["bid"] > 0) & (puts["ask"] > 0)
    puts["premium"] = np.where(has_live_quote, puts["ask"], puts["lastPrice"])
    puts["quote_source"] = np.where(has_live_quote, "ask ao vivo", "último negócio")
    mid_price = (puts["bid"] + puts["ask"]) / 2
    puts["bid_ask_spread_pct"] = np.where(has_live_quote, (puts["ask"] - puts["bid"]) / mid_price, np.nan)

    is_liquid = puts["openInterest"].fillna(0) >= settings.minimum_open_interest
    spread_ok = ~has_live_quote | (puts["bid_ask_spread_pct"] <= settings.maximum_bid_ask_spread_pct)
    puts = puts[is_liquid & spread_ok & (puts["premium"] > 0)]
    if puts.empty:
        raise MarketDataError("nenhuma put líquida na faixa de strikes")
    columns = ["contractSymbol", "strike", "premium", "quote_source", "bid", "ask", "lastPrice",
               "bid_ask_spread_pct", "openInterest", "volume", "impliedVolatility"]
    return puts[columns].sort_values("strike").reset_index(drop=True)

## 5. As contas da proteção para cada strike

Para cada put líquida, o notebook calcula a proteção da **posição inteira**. A quantidade de ações é sempre múltipla de 100, porque um contrato de opção cobre 100 ações.

In [ ]:
@dataclass
class ProtectivePutPlan:
    ticker: str
    contract_symbol: str
    expiration: pd.Timestamp
    days_to_expiration: int
    stock_price: float
    strike: float
    put_premium: float
    quote_source: str
    shares: int
    contracts: int
    cost_per_share: float
    total_cost: float
    guaranteed_sale_price: float
    guaranteed_sale_value: float
    max_loss_per_share: float
    max_loss_total: float
    max_loss_pct: float
    breakeven_price: float
    breakeven_change_pct: float
    protection_cost_pct: float
    annualized_protection_cost_pct: float
    implied_volatility: float
    open_interest: int


def shares_for_ticker(ticker: str, settings: AnalysisSettings) -> int:
    # Quantidade de ações: a definida para o ativo (arredondada ao lote) ou contratos padrão × 100.
    lot = settings.shares_per_option_contract
    if ticker in settings.shares_per_ticker:
        return int(settings.shares_per_ticker[ticker] // lot * lot)
    return settings.contracts_per_ticker * lot


def build_protective_put_plan(ticker: str, put_row: pd.Series, stock_price: float, expiration: pd.Timestamp,
                              shares: int, settings: AnalysisSettings) -> ProtectivePutPlan:
    contracts = shares // settings.shares_per_option_contract
    option_fee_per_share = settings.option_commission_per_contract / settings.shares_per_option_contract
    cost_per_share = stock_price * (1 + settings.stock_commission_pct) + put_row["premium"] + option_fee_per_share

    strike = float(put_row["strike"])
    net_sale_at_strike = strike * (1 - settings.stock_commission_pct)
    max_loss_per_share = cost_per_share - net_sale_at_strike
    breakeven_price = cost_per_share / (1 - settings.stock_commission_pct)
    days = max((expiration - TODAY).days, 1)
    protection_cost_pct = put_row["premium"] / stock_price

    return ProtectivePutPlan(
        ticker=ticker, contract_symbol=put_row["contractSymbol"], expiration=expiration, days_to_expiration=days,
        stock_price=stock_price, strike=strike, put_premium=float(put_row["premium"]),
        quote_source=put_row["quote_source"], shares=shares, contracts=contracts,
        cost_per_share=cost_per_share, total_cost=cost_per_share * shares,
        guaranteed_sale_price=strike, guaranteed_sale_value=net_sale_at_strike * shares,
        max_loss_per_share=max_loss_per_share, max_loss_total=max_loss_per_share * shares,
        max_loss_pct=max_loss_per_share / cost_per_share,
        breakeven_price=breakeven_price, breakeven_change_pct=breakeven_price / stock_price - 1,
        protection_cost_pct=protection_cost_pct,
        annualized_protection_cost_pct=protection_cost_pct * 365 / days,
        implied_volatility=float(put_row["impliedVolatility"]), open_interest=int(put_row["openInterest"]),
    )


def profit_per_share_at_expiration(plan: ProtectivePutPlan, prices_at_expiration: np.ndarray,
                                   settings: AnalysisSettings) -> np.ndarray:
    # Resultado por ação no vencimento: vende pela strike se o preço estiver abaixo dela, senão pelo mercado.
    sale_price = np.maximum(prices_at_expiration, plan.strike)
    return sale_price * (1 - settings.stock_commission_pct) - plan.cost_per_share


def stock_only_profit_per_share(stock_price: float, prices_at_expiration: np.ndarray,
                                settings: AnalysisSettings) -> np.ndarray:
    cost = stock_price * (1 + settings.stock_commission_pct)
    return prices_at_expiration * (1 - settings.stock_commission_pct) - cost

## 6. Risco com Monte Carlo (GARCH-t)

Os retornos de ações têm duas características que o modelo normal ignora: a **volatilidade muda com o tempo** (períodos calmos e agitados se alternam) e **quedas extremas acontecem mais do que o esperado** (caudas gordas). O modelo **GARCH(1,1) com distribuição t** captura as duas.

1. Ajustamos o GARCH-t ao **histórico real** de retornos diários do ativo.
2. Simulamos `monte_carlo_paths` trajetórias até o vencimento da put, **partindo da volatilidade atual**.
3. O retorno médio simulado é igual aos juros menos os dividendos, então o modelo **não aposta em alta nem em queda**. As diferenças entre as strikes vêm só do preço da put e da forma da proteção.

**Métricas** (em % do valor investido):
- **P(perda):** chance de terminar no prejuízo.
- **VaR 95%:** a perda que só é superada em 5% dos cenários.
- **ES 97,5%:** a perda média nos 2,5% piores cenários.
- **P(exercer a put):** chance de o preço terminar abaixo da strike.
- **Custo líquido esperado da proteção:** prêmio pago menos o valor médio que a put devolve nos cenários simulados. Se for positivo, a put está "cara" em relação ao risco que o modelo prevê.

In [ ]:
@dataclass
class GarchParameters:
    mean_daily_return: float
    omega: float
    alpha: float
    beta: float
    degrees_of_freedom: float
    next_day_variance: float

    @property
    def current_annual_volatility(self) -> float:
        return float(np.sqrt(self.next_day_variance * TRADING_DAYS_PER_YEAR))


def fit_garch_t(daily_log_returns: np.ndarray) -> GarchParameters:
    # Máxima verossimilhança do GARCH(1,1) com inovações t de Student.
    returns_pct = daily_log_returns * 100

    def unpack(raw):
        mean, log_omega, alpha_logit, beta_logit, log_dof = raw
        alpha = 0.3 / (1 + np.exp(-alpha_logit))
        beta = (0.999 - alpha) / (1 + np.exp(-beta_logit))
        return mean, np.exp(log_omega), alpha, beta, 2.1 + np.exp(log_dof)

    def conditional_variances(mean, omega, alpha, beta):
        shocks = returns_pct - mean
        initial_variance = shocks.var()
        drivers = omega + alpha * np.concatenate([[initial_variance], shocks[:-1] ** 2])
        variances, _ = signal.lfilter([1.0], [1.0, -beta], drivers, zi=[beta * initial_variance])
        return shocks, np.maximum(variances, 1e-8)

    def negative_log_likelihood(raw):
        mean, omega, alpha, beta, dof = unpack(raw)
        shocks, variances = conditional_variances(mean, omega, alpha, beta)
        log_density = (gammaln((dof + 1) / 2) - gammaln(dof / 2) - 0.5 * np.log((dof - 2) * np.pi * variances)
                       - (dof + 1) / 2 * np.log1p(shocks ** 2 / ((dof - 2) * variances)))
        return -log_density.sum()

    start = [returns_pct.mean(), np.log(returns_pct.var() * 0.05), -1.0, 2.0, np.log(4.0)]
    solution = optimize.minimize(negative_log_likelihood, start, method="Nelder-Mead",
                                 options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-6})
    mean, omega, alpha, beta, dof = unpack(solution.x)
    shocks, variances = conditional_variances(mean, omega, alpha, beta)
    next_variance = omega + alpha * shocks[-1] ** 2 + beta * variances[-1]
    return GarchParameters(mean / 100, omega / 1e4, alpha, beta, dof, next_variance / 1e4)


def simulate_prices_at_expiration(garch: GarchParameters, start_price: float, trading_days: int,
                                  annual_drift: float, number_of_paths: int, rng: np.random.Generator) -> np.ndarray:
    # Preço final de cada trajetória; o drift é ajustado para E[preço final] = preço inicial × e^(drift × prazo).
    variance = np.full(number_of_paths, garch.next_day_variance)
    standardized = np.sqrt((garch.degrees_of_freedom - 2) / garch.degrees_of_freedom)
    cumulative_log_return = np.zeros(number_of_paths)
    for _ in range(trading_days):
        shock = np.sqrt(variance) * rng.standard_t(garch.degrees_of_freedom, number_of_paths) * standardized
        cumulative_log_return += shock
        variance = garch.omega + garch.alpha * shock ** 2 + garch.beta * variance
    centered = cumulative_log_return - np.log(np.mean(np.exp(cumulative_log_return)))
    return start_price * np.exp(centered + annual_drift * trading_days / TRADING_DAYS_PER_YEAR)


def risk_metrics(profit_per_share: np.ndarray, cost_per_share: float) -> dict:
    loss_pct = -profit_per_share / cost_per_share
    var_975 = np.percentile(loss_pct, 97.5)
    return {"P(perda)": float((profit_per_share < 0).mean()),
            "resultado esperado %": float(-loss_pct.mean()),
            "VaR 95%": float(np.percentile(loss_pct, 95)),
            "ES 97,5%": float(loss_pct[loss_pct >= var_975].mean())}

## 7. Rodando a análise para cada ativo

Para cada ativo: baixa os dados, monta o menu de strikes, roda o Monte Carlo e escolhe a strike recomendada. Quando algo falta no Yahoo, o ativo fica **não elegível** e o motivo é registrado.

**Regra de recomendação.** Entre as strikes cuja **perda máxima** fica abaixo de `max_acceptable_loss_pct`, escolhe a de **menor custo líquido esperado da proteção**. Se nenhuma cumprir o limite, escolhe a de menor perda máxima e avisa.

In [ ]:
def build_strike_menu(plans: list, simulated_prices: np.ndarray, settings: AnalysisSettings) -> pd.DataFrame:
    stock_price = plans[0].stock_price
    unprotected = risk_metrics(stock_only_profit_per_share(stock_price, simulated_prices, settings),
                               stock_price * (1 + settings.stock_commission_pct))
    rows = []
    for plan in plans:
        protected = risk_metrics(profit_per_share_at_expiration(plan, simulated_prices, settings), plan.cost_per_share)
        expected_put_payoff = np.maximum(plan.strike - simulated_prices, 0).mean()
        rows.append({**asdict(plan), **{f"{name} (com put)": value for name, value in protected.items()},
                     "P(exercer a put)": float((simulated_prices < plan.strike).mean()),
                     "custo líquido esperado %": (plan.put_premium - expected_put_payoff) / stock_price,
                     "redução do ES (p.p.)": unprotected["ES 97,5%"] - protected["ES 97,5%"]})
    menu = pd.DataFrame(rows)
    for name, value in unprotected.items():
        menu[f"{name} (sem put)"] = value
    return menu


def recommend_strike(menu: pd.DataFrame, settings: AnalysisSettings) -> tuple:
    within_limit = menu[menu["max_loss_pct"] <= settings.max_acceptable_loss_pct]
    if len(within_limit):
        return within_limit.loc[within_limit["custo líquido esperado %"].idxmin()], True
    return menu.loc[menu["max_loss_pct"].idxmin()], False


def analyze_ticker(ticker: str, risk_free_rate: float, settings: AnalysisSettings) -> dict:
    closes = download_price_history(ticker, settings.history_start_date)
    stock_price = get_current_stock_price(ticker, float(closes.iloc[-1]))
    expiration = choose_expiration(list_option_expirations(ticker), settings)
    shares = shares_for_ticker(ticker, settings)
    if shares < settings.shares_per_option_contract:
        raise MarketDataError(f"quantidade definida menor que 1 contrato ({settings.shares_per_option_contract} ações)")
    required_capital = shares * stock_price
    if settings.max_capital_per_ticker and required_capital > settings.max_capital_per_ticker:
        raise MarketDataError(f"exige {required_capital:,.0f} para {shares} ações, acima do limite de capital")
    liquid_puts = prepare_liquid_puts(download_put_chain(ticker, expiration), stock_price, settings)
    plans = [build_protective_put_plan(ticker, row, stock_price, expiration, shares, settings)
             for _, row in liquid_puts.iterrows()]

    daily_log_returns = np.log(closes).diff().dropna().values
    garch = fit_garch_t(daily_log_returns)
    dividend_yield = get_trailing_dividend_yield(ticker, stock_price)
    trading_days = max(int(round(plans[0].days_to_expiration * TRADING_DAYS_PER_YEAR / 365)), 1)
    rng = np.random.default_rng(settings.random_seed)
    simulated_prices = simulate_prices_at_expiration(garch, stock_price, trading_days,
                                                     risk_free_rate - dividend_yield, settings.monte_carlo_paths, rng)
    menu = build_strike_menu(plans, simulated_prices, settings)
    recommended, respects_limit = recommend_strike(menu, settings)
    return {"closes": closes, "stock_price": stock_price, "shares": shares, "expiration": expiration,
            "garch": garch, "dividend_yield": dividend_yield, "simulated_prices": simulated_prices,
            "liquid_puts": liquid_puts, "menu": menu, "recommended": recommended,
            "recommended_respects_limit": respects_limit, "uses_last_trade_prices": bool((liquid_puts["quote_source"] == "último negócio").any())}


RISK_FREE_RATE = get_us_risk_free_rate()
print(f"Taxa livre de risco (^IRX, Yahoo): {RISK_FREE_RATE:.2%} ao ano")

RESULTS, NOT_ELIGIBLE = {}, {}
for ticker in SETTINGS.tickers:
    try:
        RESULTS[ticker] = analyze_ticker(ticker, RISK_FREE_RATE, SETTINGS)
    except MarketDataError as reason:
        NOT_ELIGIBLE[ticker] = str(reason)
    except Exception as unexpected:
        NOT_ELIGIBLE[ticker] = f"erro ao baixar ou processar: {unexpected}"

print(f"Elegíveis: {len(RESULTS)} de {len(SETTINGS.tickers)}")
if NOT_ELIGIBLE:
    display(pd.Series(NOT_ELIGIBLE, name="motivo").to_frame())
if any(result["uses_last_trade_prices"] for result in RESULTS.values()):
    display(Markdown("> **Atenção:** algumas puts estão com bid/ask zerados (mercado fechado?) e usam o **último negócio**. "
                     "Rode de novo durante o pregão para ter preços executáveis."))

## 8. O menu de proteção de cada ativo

Cada linha é uma strike real da cadeia do Yahoo. A linha marcada com ► é a recomendada. Para ler:
- **Venda garantida:** o valor mínimo que você recebe pela posição inteira, até o vencimento.
- **Perda máxima:** quanto você pode perder no pior caso. Já inclui o prêmio e as taxas.
- **Break-even:** o preço no vencimento para empatar. Acima dele, lucro.
- **ES com/sem put:** a perda média nos piores 2,5% dos cenários do Monte Carlo.

In [ ]:
MENU_COLUMNS = {
    "strike": "strike (venda garantida/ação)", "put_premium": "prêmio da put", "quote_source": "preço usado",
    "guaranteed_sale_value": "venda garantida (posição)", "total_cost": "custo total", "max_loss_total": "perda máx.",
    "max_loss_pct": "perda máx. %", "breakeven_price": "break-even", "breakeven_change_pct": "break-even vs. hoje",
    "protection_cost_pct": "custo da proteção", "annualized_protection_cost_pct": "custo anualizado",
    "P(exercer a put)": "P(exercer)", "P(perda) (com put)": "P(perda) com put",
    "ES 97,5% (sem put)": "ES sem put", "ES 97,5% (com put)": "ES com put",
    "custo líquido esperado %": "custo líq. esperado",
}
MENU_FORMATS = {
    "strike (venda garantida/ação)": "{:,.2f}", "prêmio da put": "{:,.2f}", "venda garantida (posição)": "{:,.0f}",
    "custo total": "{:,.0f}", "perda máx.": "{:,.0f}", "perda máx. %": "{:.1%}", "break-even": "{:,.2f}",
    "break-even vs. hoje": "{:+.1%}", "custo da proteção": "{:.2%}", "custo anualizado": "{:.1%}",
    "P(exercer)": "{:.0%}", "P(perda) com put": "{:.0%}", "ES sem put": "{:.1%}", "ES com put": "{:.1%}",
    "custo líq. esperado": "{:+.2%}",
}

for ticker, result in RESULTS.items():
    menu = result["menu"]
    recommended_strike = result["recommended"]["strike"]
    table = menu[[column for column in MENU_COLUMNS if column in menu.columns]].rename(columns=MENU_COLUMNS)
    table.index = ["► recomendada" if strike == recommended_strike else "" for strike in menu["strike"]]
    contracts = result['shares'] // SETTINGS.shares_per_option_contract
    display(Markdown(f"### {ticker} · preço {result['stock_price']:,.2f} · {result['shares']} ações "
                     f"({contracts} {'contrato' if contracts == 1 else 'contratos'}) · vencimento "
                     f"{result['expiration'].date()} · vol. atual (GARCH) {result['garch'].current_annual_volatility:.1%}"))
    display(table.style.format(MENU_FORMATS))
    if not result["recommended_respects_limit"]:
        display(Markdown(f"> Nenhuma strike deixa a perda máxima abaixo de {SETTINGS.max_acceptable_loss_pct:.0%}; "
                         "a recomendada é a de menor perda máxima."))

In [ ]:
def plot_strike_menu(ticker: str, result: dict) -> None:
    # Cada ponto é uma strike: quanto custa (x) contra quanto você pode perder no máximo (y).
    menu = result["menu"]
    figure, axis = plt.subplots(figsize=(10, 4.2))
    axis.plot(menu["protection_cost_pct"], menu["max_loss_pct"], "-", color=COLORS["muted"], lw=1)
    axis.plot(menu["protection_cost_pct"], menu["max_loss_pct"], "o", color=COLORS["stock"], ms=7,
              mec=COLORS["surface"], mew=1.5)
    chosen = result["recommended"]
    axis.plot(chosen["protection_cost_pct"], chosen["max_loss_pct"], "o", color=COLORS["protected"], ms=12,
              mec=COLORS["surface"], mew=2)
    axis.annotate(f"recomendada: strike {chosen['strike']:,.2f}", (chosen["protection_cost_pct"], chosen["max_loss_pct"]),
                  xytext=(-10, 10), textcoords="offset points", ha="right", color=COLORS["ink_strong"])
    axis.axhline(SETTINGS.max_acceptable_loss_pct, color=COLORS["loss"], lw=1, ls="--")
    axis.annotate(f"perda máxima aceitável ({SETTINGS.max_acceptable_loss_pct:.0%})",
                  (axis.get_xlim()[1], SETTINGS.max_acceptable_loss_pct), xytext=(-4, 4), textcoords="offset points",
                  ha="right", color=COLORS["loss"])
    axis.xaxis.set_major_formatter(PercentFormatter(1.0)); axis.yaxis.set_major_formatter(PercentFormatter(1.0))
    axis.set_xlabel("custo da proteção (prêmio ÷ preço da ação)")
    axis.set_ylabel("perda máxima da posição protegida")
    axis.set_title(f"{ticker}: quanto custa cada nível de proteção (puts reais, vencimento {result['expiration'].date()})")
    plt.show()


def plot_protection_payoff(ticker: str, result: dict) -> None:
    # Resultado da posição no vencimento, com e sem put, e a distribuição de preços simulada.
    chosen = result["recommended"]
    plan = ProtectivePutPlan(**{key: chosen[key] for key in ProtectivePutPlan.__dataclass_fields__})
    prices = result["simulated_prices"]
    low, high = np.percentile(prices, [0.5, 99.5])
    price_grid = np.linspace(min(low, plan.strike * 0.92), high, 500)
    shares = plan.shares
    protected = profit_per_share_at_expiration(plan, price_grid, SETTINGS) * shares
    unprotected = stock_only_profit_per_share(plan.stock_price, price_grid, SETTINGS) * shares

    figure, (payoff_axis, distribution_axis) = plt.subplots(2, 1, figsize=(10, 6.4), sharex=True,
                                                            gridspec_kw={"height_ratios": [2.2, 1]})
    payoff_axis.plot(price_grid, unprotected, color=COLORS["stock"], label="só a ação")
    payoff_axis.plot(price_grid, protected, color=COLORS["protected"], label=f"ação + put {plan.strike:,.2f}")
    payoff_axis.axhline(0, color=COLORS["ink"], lw=1)
    payoff_axis.axhline(-plan.max_loss_total, color=COLORS["loss"], lw=1, ls="--")
    payoff_axis.annotate(f"perda máxima {plan.max_loss_total:,.0f}", (price_grid[0], -plan.max_loss_total),
                         xytext=(2, -14), textcoords="offset points", color=COLORS["loss"])
    payoff_axis.axvline(plan.strike, color=COLORS["ink"], lw=1, ls="--")
    payoff_axis.annotate(f"venda garantida {plan.strike:,.2f}", (plan.strike, payoff_axis.get_ylim()[1]),
                         xytext=(4, -14), textcoords="offset points", color=COLORS["ink"])
    payoff_axis.plot(plan.breakeven_price, 0, "o", color=COLORS["protected"], ms=8, mec=COLORS["surface"], mew=2)
    payoff_axis.annotate(f"break-even {plan.breakeven_price:,.2f}", (plan.breakeven_price, 0), xytext=(6, -16),
                         textcoords="offset points", color=COLORS["ink"])
    payoff_axis.set_ylabel("resultado da posição no vencimento")
    payoff_axis.set_title(f"{ticker}: {shares} ações + {plan.contracts} put(s), vencimento {plan.expiration.date()}")
    payoff_axis.legend(loc="upper left")

    distribution_axis.hist(prices, bins=120, range=(price_grid[0], price_grid[-1]), color=COLORS["stock"], alpha=0.5)
    distribution_axis.axvline(plan.stock_price, color=COLORS["ink"], lw=1, ls=":")
    distribution_axis.set_yticks([]); distribution_axis.grid(axis="y", visible=False)
    distribution_axis.set_xlabel("preço da ação no vencimento; abaixo: distribuição simulada pelo Monte Carlo (GARCH-t)")
    plt.show()


for ticker, result in RESULTS.items():
    plot_strike_menu(ticker, result)
    plot_protection_payoff(ticker, result)

## 9. Deep Q-Network para o melhor dia de entrada

**A pergunta:** você decidiu comprar a ação (com a put). Vale comprar **hoje** ou esperar alguns dias por um preço melhor, até no máximo `entry_window_days` dias úteis?

**Como o agente funciona**
- A cada dia da janela, ele vê o **estado do mercado** e escolhe entre **comprar agora** ou **esperar**. No último dia da janela, a compra é obrigatória.
- **Recompensa:** a economia em relação a ter comprado no primeiro dia da janela, `(preço no 1º dia − preço pago) ÷ preço no 1º dia`. Esperar e pagar mais caro dá recompensa negativa.
- **Estado:** sete sinais calculados só com dados do passado, mais o tempo que resta na janela:
  - retorno dos últimos 5 e 20 dias;
  - distância do preço às médias de 20 e 50 dias;
  - RSI de 14 dias;
  - volatilidade recente (20 dias) em relação à de 6 meses;
  - queda em relação à máxima de 60 dias.
- **Rede:** MLP com duas camadas ocultas, escrita em NumPy, com *experience replay* e rede-alvo (Mnih et al., 2015). O alvo segue a variante **Double DQN** (van Hasselt et al., 2016), que corrige a tendência da DQN de superestimar o valor de esperar.

**Avaliação honesta**
- Treina só com os dados **antigos** (os primeiros 70% do histórico) e testa nos **30% mais recentes**, que a rede nunca viu, com uma janela de separação entre os dois.
- As janelas de teste não se sobrepõem, para não inflar a estatística.
- Compara com três referências: **comprar no 1º dia**, **comprar um pouco por dia** (preço médio da janela) e **um dia aleatório**.
- Só trate o sinal de hoje como útil se a DQN tiver batido as referências **com significância** no teste.

In [ ]:
ENTRY_SIGNAL_NAMES = ["retorno 5 dias", "retorno 20 dias", "distância da média 20", "distância da média 50",
                      "RSI 14", "vol. 20d / vol. 6 meses", "queda desde a máxima 60d"]
ACTION_WAIT, ACTION_BUY = 0, 1


def compute_entry_signals(closes: pd.Series) -> pd.DataFrame:
    # Sinais de mercado usando apenas informação disponível até cada dia.
    log_returns = np.log(closes).diff()
    price_changes = closes.diff()
    average_gain = price_changes.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    average_loss = (-price_changes.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    signals = pd.DataFrame({
        "retorno 5 dias": closes / closes.shift(5) - 1,
        "retorno 20 dias": closes / closes.shift(20) - 1,
        "distância da média 20": closes / closes.rolling(20).mean() - 1,
        "distância da média 50": closes / closes.rolling(50).mean() - 1,
        "RSI 14": 100 - 100 / (1 + average_gain / average_loss.replace(0, np.nan)),
        "vol. 20d / vol. 6 meses": log_returns.rolling(20).std() / log_returns.rolling(126).std(),
        "queda desde a máxima 60d": closes / closes.rolling(60).max() - 1,
    })
    return signals.dropna()


class QNetwork:
    # Rede neural pequena (2 camadas ocultas ReLU) que estima o valor de cada ação; treino com Adam.
    def __init__(self, input_size: int, output_size: int, hidden_units: int, learning_rate: float, seed: int):
        rng = np.random.default_rng(seed)
        self.weights = [rng.normal(0, np.sqrt(2 / input_size), (input_size, hidden_units)), np.zeros(hidden_units),
                        rng.normal(0, np.sqrt(2 / hidden_units), (hidden_units, hidden_units)), np.zeros(hidden_units),
                        rng.normal(0, 0.01, (hidden_units, output_size)), np.zeros(output_size)]
        self.first_moments = [np.zeros_like(w) for w in self.weights]
        self.second_moments = [np.zeros_like(w) for w in self.weights]
        self.learning_rate, self.update_count = learning_rate, 0

    def _forward(self, states):
        w1, b1, w2, b2, w3, b3 = self.weights
        hidden1_pre = states @ w1 + b1; hidden1 = np.maximum(hidden1_pre, 0)
        hidden2_pre = hidden1 @ w2 + b2; hidden2 = np.maximum(hidden2_pre, 0)
        return hidden2 @ w3 + b3, (states, hidden1_pre, hidden1, hidden2_pre, hidden2)

    def action_values(self, states: np.ndarray) -> np.ndarray:
        return self._forward(states)[0]

    def train_step(self, states, actions, targets) -> float:
        predictions, (inputs, hidden1_pre, hidden1, hidden2_pre, hidden2) = self._forward(states)
        batch = np.arange(len(states))
        errors = np.clip(predictions[batch, actions] - targets, -1, 1)     # gradiente da perda de Huber
        output_grad = np.zeros_like(predictions); output_grad[batch, actions] = errors / len(states)
        hidden2_grad = (output_grad @ self.weights[4].T) * (hidden2_pre > 0)
        hidden1_grad = (hidden2_grad @ self.weights[2].T) * (hidden1_pre > 0)
        gradients = [inputs.T @ hidden1_grad, hidden1_grad.sum(0), hidden1.T @ hidden2_grad, hidden2_grad.sum(0),
                     hidden2.T @ output_grad, output_grad.sum(0)]
        self.update_count += 1
        for index, gradient in enumerate(gradients):
            self.first_moments[index] = 0.9 * self.first_moments[index] + 0.1 * gradient
            self.second_moments[index] = 0.999 * self.second_moments[index] + 0.001 * gradient ** 2
            corrected_first = self.first_moments[index] / (1 - 0.9 ** self.update_count)
            corrected_second = self.second_moments[index] / (1 - 0.999 ** self.update_count)
            self.weights[index] -= self.learning_rate * corrected_first / (np.sqrt(corrected_second) + 1e-8)
        return float((errors ** 2).mean())


class EntryTimingEnvironment:
    # Episódios de compra: começam num dia do histórico e terminam quando o agente compra (ou a janela acaba).
    def __init__(self, prices: np.ndarray, normalized_signals: np.ndarray, window_days: int):
        self.prices, self.signals, self.window_days = prices, normalized_signals, window_days

    def states(self, start_days: np.ndarray, days_elapsed: int) -> np.ndarray:
        time_left = np.full((len(start_days), 1), 1 - days_elapsed / self.window_days)
        return np.hstack([self.signals[start_days + days_elapsed], time_left])

    def savings_pct(self, start_days: np.ndarray, purchase_days: np.ndarray) -> np.ndarray:
        first_day_price = self.prices[start_days]
        return 100 * (first_day_price - self.prices[purchase_days]) / first_day_price


def train_entry_dqn(environment: EntryTimingEnvironment, training_start_days: np.ndarray,
                    settings: AnalysisSettings, seed: int) -> QNetwork:
    rng = np.random.default_rng(seed)
    state_size = environment.signals.shape[1] + 1
    online_network = QNetwork(state_size, 2, settings.dqn_hidden_units, 1e-3, seed)
    target_network = copy.deepcopy(online_network)
    capacity, parallel_episodes, batch_size = 100_000, 64, 256
    memory_states = np.zeros((capacity, state_size)); memory_next_states = np.zeros((capacity, state_size))
    memory_actions = np.zeros(capacity, int); memory_rewards = np.zeros(capacity); memory_done = np.zeros(capacity)
    memory_next_is_last_day = np.zeros(capacity, bool)   # no último dia só existe "comprar"
    write_position, stored = 0, 0

    for training_round in range(settings.dqn_training_rounds):
        exploration_rate = max(0.05, 1 - training_round / (0.7 * settings.dqn_training_rounds))
        start_days = rng.choice(training_start_days, parallel_episodes)
        still_waiting = np.ones(parallel_episodes, bool)
        for days_elapsed in range(environment.window_days):
            active = np.where(still_waiting)[0]
            if not len(active):
                break
            states = environment.states(start_days[active], days_elapsed)
            greedy = online_network.action_values(states).argmax(1)
            explore = rng.random(len(active)) < exploration_rate
            actions = np.where(explore, rng.integers(2, size=len(active)), greedy)
            last_day = days_elapsed == environment.window_days - 1
            if last_day:
                actions[:] = ACTION_BUY
            bought = actions == ACTION_BUY
            rewards = np.where(bought, environment.savings_pct(start_days[active], start_days[active] + days_elapsed), 0.0)
            next_states = states if last_day else environment.states(start_days[active], days_elapsed + 1)
            slots = (write_position + np.arange(len(active))) % capacity
            memory_states[slots], memory_actions[slots], memory_rewards[slots] = states, actions, rewards
            memory_next_states[slots], memory_done[slots] = next_states, bought.astype(float)
            memory_next_is_last_day[slots] = days_elapsed + 1 == environment.window_days - 1
            write_position = (write_position + len(active)) % capacity
            stored = min(stored + len(active), capacity)
            still_waiting[active[bought]] = False

            if stored >= 1_000:
                for _ in range(4):
                    sample = rng.integers(stored, size=batch_size)
                    # Double DQN: a rede online escolhe a próxima ação e a rede-alvo avalia (reduz o otimismo do max)
                    next_states = memory_next_states[sample]
                    best_next_actions = online_network.action_values(next_states).argmax(1)
                    best_next_actions[memory_next_is_last_day[sample]] = ACTION_BUY
                    next_values = target_network.action_values(next_states)[np.arange(batch_size), best_next_actions]
                    targets = memory_rewards[sample] + (1 - memory_done[sample]) * next_values
                    online_network.train_step(memory_states[sample], memory_actions[sample], targets)
                    if online_network.update_count % 200 == 0:
                        target_network = copy.deepcopy(online_network)
    return online_network


def greedy_purchase_days(network: QNetwork, environment: EntryTimingEnvironment, start_days: np.ndarray) -> np.ndarray:
    # Para cada início de janela, o dia em que a rede treinada compraria.
    purchase_days = start_days + environment.window_days - 1
    decided = np.zeros(len(start_days), bool)
    for days_elapsed in range(environment.window_days - 1):
        undecided = np.where(~decided)[0]
        values = network.action_values(environment.states(start_days[undecided], days_elapsed))
        buys_now = undecided[values[:, ACTION_BUY] >= values[:, ACTION_WAIT]]
        purchase_days[buys_now] = start_days[buys_now] + days_elapsed
        decided[buys_now] = True
    return purchase_days


def bootstrap_mean_interval(values: np.ndarray, rng: np.random.Generator, draws: int = 5_000) -> tuple:
    means = rng.choice(values, (draws, len(values))).mean(1)
    return tuple(np.percentile(means, [2.5, 97.5]))

In [ ]:
def evaluate_entry_timing(ticker: str, closes: pd.Series, settings: AnalysisSettings) -> dict:
    signals = compute_entry_signals(closes)
    prices = closes.reindex(signals.index).values
    window = settings.entry_window_days
    total_days = len(prices)
    test_start = int(total_days * (1 - settings.test_period_fraction))
    training_end = test_start - window                      # separação entre treino e teste

    signal_mean = signals.iloc[:training_end].mean()
    signal_std = signals.iloc[:training_end].std().replace(0, 1)
    normalized = ((signals - signal_mean) / signal_std).clip(-5, 5).values
    environment = EntryTimingEnvironment(prices, normalized, window)

    network = train_entry_dqn(environment, np.arange(0, training_end - window), settings, settings.random_seed)

    rng = np.random.default_rng(settings.random_seed + 1)
    test_start_days = np.arange(test_start, total_days - window, window)        # janelas sem sobreposição
    dqn_days = greedy_purchase_days(network, environment, test_start_days)
    random_days = test_start_days + rng.integers(window, size=len(test_start_days))
    window_average = np.array([prices[day:day + window].mean() for day in test_start_days])
    savings = {
        "DQN": environment.savings_pct(test_start_days, dqn_days),
        "Comprar no 1º dia": np.zeros(len(test_start_days)),
        "Preço médio da janela": 100 * (prices[test_start_days] - window_average) / prices[test_start_days],
        "Dia aleatório": environment.savings_pct(test_start_days, random_days),
    }
    summary_rows = {}
    for name, values in savings.items():
        lower, upper = bootstrap_mean_interval(values, rng)
        summary_rows[name] = {"economia média (%)": values.mean(), "IC 95% inferior": lower,
                              "IC 95% superior": upper, "% janelas melhores que o 1º dia": (values > 0).mean()}
    summary = pd.DataFrame(summary_rows).T
    p_value_vs_first_day = stats.ttest_1samp(savings["DQN"], 0).pvalue
    p_value_vs_average = stats.ttest_1samp(savings["DQN"] - savings["Preço médio da janela"], 0).pvalue
    beats_benchmarks = (savings["DQN"].mean() > 0 and p_value_vs_first_day < 0.05
                        and savings["DQN"].mean() > savings["Preço médio da janela"].mean() and p_value_vs_average < 0.05)

    today_state = np.hstack([normalized[-1], [1.0]])[None, :]
    wait_value, buy_value = network.action_values(today_state)[0]
    return {"summary": summary, "test_windows": len(test_start_days), "average_purchase_day": float((dqn_days - test_start_days).mean()),
            "p_value_vs_first_day": p_value_vs_first_day, "p_value_vs_average": p_value_vs_average,
            "beats_benchmarks": bool(beats_benchmarks), "today_action": "comprar hoje" if buy_value >= wait_value else "esperar",
            "today_expected_gain_from_waiting_pct": float(wait_value - buy_value),
            "test_period_start": signals.index[test_start].date(), "network": network}


def plot_entry_evaluation(ticker: str, evaluation: dict) -> None:
    summary = evaluation["summary"].iloc[::-1]
    figure, axis = plt.subplots(figsize=(10, 3.4))
    for position, (name, row) in enumerate(summary.iterrows()):
        color = COLORS["protected"] if name == "DQN" else COLORS["stock"]
        axis.plot([row["IC 95% inferior"], row["IC 95% superior"]], [position, position], color=color, lw=2)
        axis.plot(row["economia média (%)"], position, "o", color=color, ms=8, mec=COLORS["surface"], mew=2)
        axis.annotate(f"{row['economia média (%)']:+.2f}%", (row["IC 95% superior"], position), xytext=(6, -4),
                      textcoords="offset points", color=COLORS["ink"])
    axis.axvline(0, color=COLORS["ink"], lw=1)
    axis.set_yticks(range(len(summary)), summary.index); axis.grid(axis="y", visible=False)
    axis.set_xlabel("economia média por compra vs. comprar no 1º dia (%), com intervalo de confiança de 95%")
    axis.set_title(f"{ticker}: ponto de entrada no período de teste ({evaluation['test_windows']} janelas desde {evaluation['test_period_start']})")
    plt.show()


ENTRY_EVALUATIONS = {}
for ticker, result in RESULTS.items():
    evaluation = evaluate_entry_timing(ticker, result["closes"], SETTINGS)
    ENTRY_EVALUATIONS[ticker] = evaluation
    display(Markdown(f"### {ticker}"))
    display(evaluation["summary"].style.format({"economia média (%)": "{:+.2f}", "IC 95% inferior": "{:+.2f}",
                                                "IC 95% superior": "{:+.2f}", "% janelas melhores que o 1º dia": "{:.0%}"}))
    plot_entry_evaluation(ticker, evaluation)
    verdict = "**bateu** as referências com significância" if evaluation["beats_benchmarks"] else "**não bateu** as referências com significância"
    display(Markdown(f"A DQN {verdict} (p-valor vs. 1º dia = {evaluation['p_value_vs_first_day']:.2f}; vs. preço médio = "
                     f"{evaluation['p_value_vs_average']:.2f}). Em média comprou no dia {evaluation['average_purchase_day']:.1f} da janela. "
                     f"Sinal de hoje: **{evaluation['today_action']}**."))

## 10. Painel final

Uma linha por ativo elegível, com tudo o que você precisa para decidir: **quando entrar** (DQN), **qual put comprar** (strike, vencimento e código do contrato), **por quanto você consegue vender no pior caso**, **quanto pode perder no máximo** e **a partir de que preço começa o lucro**.

In [ ]:
def describe_entry_signal(evaluation: dict) -> str:
    if not evaluation["beats_benchmarks"]:
        return f"{evaluation['today_action']} (sinal fraco: a DQN não bateu as referências no teste)"
    if evaluation["today_action"] == "esperar":
        return f"esperar (ganho estimado ~{evaluation['today_expected_gain_from_waiting_pct']:.2f}%)"
    return "comprar hoje"


panel_rows = []
for ticker, result in RESULTS.items():
    chosen, evaluation = result["recommended"], ENTRY_EVALUATIONS[ticker]
    panel_rows.append({
        "ativo": ticker,
        "entrada (DQN)": describe_entry_signal(evaluation),
        "comprar": f"{chosen['shares']} ações a ~{chosen['stock_price']:,.2f} + {chosen['contracts']} put(s) {chosen['contract_symbol']}",
        "strike / vencimento": f"{chosen['strike']:,.2f} / {chosen['expiration'].date()}",
        "prêmio (preço usado)": f"{chosen['put_premium']:,.2f} ({chosen['quote_source']})",
        "custo total": f"{chosen['total_cost']:,.0f}",
        "venda garantida": f"{chosen['guaranteed_sale_value']:,.0f}",
        "perda máxima": f"{chosen['max_loss_total']:,.0f} ({chosen['max_loss_pct']:.1%})",
        "lucro acima de": f"{chosen['breakeven_price']:,.2f} ({chosen['breakeven_change_pct']:+.1%})",
        "ES 97,5% sem → com put": f"{chosen['ES 97,5% (sem put)']:.1%} → {chosen['ES 97,5% (com put)']:.1%}",
        "P(perda) com put": f"{chosen['P(perda) (com put)']:.0%}",
    })
for ticker, reason in NOT_ELIGIBLE.items():
    panel_rows.append({"ativo": ticker, "entrada (DQN)": "-", "comprar": f"não elegível: {reason}"})

display(pd.DataFrame(panel_rows).set_index("ativo").fillna(""))
for ticker, result in RESULTS.items():
    chosen = result["recommended"]
    display(Markdown(
        f"**{ticker}:** comprando {chosen['shares']} ações e {chosen['contracts']} put(s) de strike {chosen['strike']:,.2f}, "
        f"você garante vender a posição por pelo menos **{chosen['guaranteed_sale_value']:,.0f}** até {chosen['expiration'].date()}. "
        f"A perda máxima é **{chosen['max_loss_total']:,.0f}** ({chosen['max_loss_pct']:.1%}), mesmo que a ação despenque. "
        f"Para sair sem perda, a ação precisa estar em **{chosen['breakeven_price']:,.2f}** no vencimento "
        f"({chosen['breakeven_change_pct']:+.1%}); acima disso, é lucro."))

## 11. Limitações

- **Atraso e horário:** as cotações do Yahoo têm cerca de 15 minutos de atraso. Fora do pregão, bid e ask vêm zerados e o notebook usa o último negócio, que pode estar desatualizado. Confirme o ask na corretora antes de enviar a ordem.
- **Opções americanas:** opções de ações dos EUA podem ser exercidas a qualquer momento. A put dá o direito de vender pela strike antes do vencimento também, então a proteção vale durante todo o prazo, não só no fim.
- **Dividendos:** se a ação pagar dividendos durante o prazo, você os recebe por ter a ação. O Monte Carlo já desconta o dividendo do drift.
- **O Monte Carlo é um modelo:** o GARCH-t captura caudas gordas e mudança de volatilidade, mas não prevê eventos únicos. A perda máxima da put, ao contrário, é **contratual** e não depende de modelo.
- **A DQN aprende o passado:** padrões de entrada mudam. Por isso ela é testada em um período que não viu, e o painel avisa quando o sinal é fraco.
- **Impostos** não estão incluídos.

**Referências**
- Hull, J. *Options, Futures, and Other Derivatives*. Pearson. (put de proteção, cap. 12)
- Israelov, R. (2019). *Pathetic Protection: The Elusive Benefits of Protective Puts*. Journal of Alternative Investments, 21(3).
- Bollerslev, T. (1986). *Generalized Autoregressive Conditional Heteroskedasticity*. Journal of Econometrics.
- Mnih, V. et al. (2015). *Human-level control through deep reinforcement learning*. Nature.
- van Hasselt, H., Guez, A. e Silver, D. (2016). *Deep Reinforcement Learning with Double Q-learning*. AAAI.
- Nevmyvaka, Y., Feng, Y. e Kearns, M. (2006). *Reinforcement Learning for Optimized Trade Execution*. ICML.